In [1]:
import networkx as nx
import numpy as np
from gwgraphs import *

In [2]:
def read_graph_data():
    # Read the adjacency matrix (edges)
    with open("BZR_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open("BZR_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open("BZR_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open("BZR_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    # Read node attributes
    with open("BZR_node_attributes.txt", "r") as f:
        node_attributes = [
            list(map(float, line.strip().split(','))) for line in f if line.strip()
        ]

    return edges, graph_indicators, graph_labels, node_labels, node_attributes


def construct_networkx_graph(edges, graph_indicators, graph_labels,
                             node_labels, node_attributes):
    graphs = {}
    graphs_normalized = {}
    
    # Normalize all node attributes
    normalized_attributes = normalize_features(node_attributes)
    
    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])
            graphs_normalized[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[
            node1 - 1]  # Determine which graph the edge belongs to
        graphs[graph_id].add_edge(node1, node2)
        graphs_normalized[graph_id].add_edge(node1, node2)

    for node_id, (graph_id, node_label,
                  node_attr, norm_attr) in enumerate(zip(graph_indicators, node_labels,
                                              node_attributes, normalized_attributes),
                                          start=1):
        # Original graphs with original features
        graphs[graph_id].nodes[node_id]['label'] = node_label
        graphs[graph_id].nodes[node_id]['attributes'] = node_attr
        
        # Normalized graphs with normalized features
        graphs_normalized[graph_id].nodes[node_id]['label'] = node_label
        graphs_normalized[graph_id].nodes[node_id]['attributes'] = norm_attr

    return graphs, graphs_normalized


def compute_bzr_distance_matrix(graph):
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    # Get the correct node indices to be used for indexing the matrix
    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")

        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix


def normalize_features(features):
    features = np.array(features)
    # Normalize each feature dimension independently
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0)
    # Avoid division by zero
    std[std == 0] = 1
    normalized_features = (features - mean) / std
    return normalized_features.tolist()

In [3]:
edges, graph_indicators, graph_labels, node_labels, node_attributes = read_graph_data()
graphs, graphs_normalized = construct_networkx_graph(edges, graph_indicators, graph_labels,
                                                   node_labels, node_attributes)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
        )

Graph ID: 1, number of nodes: 30, label: -1
Graph ID: 2, number of nodes: 33, label: -1
Graph ID: 3, number of nodes: 30, label: -1
Graph ID: 4, number of nodes: 33, label: -1
Graph ID: 5, number of nodes: 30, label: -1
Graph ID: 6, number of nodes: 33, label: -1
Graph ID: 7, number of nodes: 30, label: -1
Graph ID: 8, number of nodes: 30, label: -1
Graph ID: 9, number of nodes: 33, label: -1
Graph ID: 10, number of nodes: 30, label: -1
Graph ID: 11, number of nodes: 30, label: -1
Graph ID: 12, number of nodes: 30, label: -1
Graph ID: 13, number of nodes: 33, label: -1
Graph ID: 14, number of nodes: 33, label: -1
Graph ID: 15, number of nodes: 33, label: -1
Graph ID: 16, number of nodes: 33, label: -1
Graph ID: 17, number of nodes: 32, label: -1
Graph ID: 18, number of nodes: 32, label: -1
Graph ID: 19, number of nodes: 32, label: -1
Graph ID: 20, number of nodes: 35, label: -1
Graph ID: 21, number of nodes: 32, label: -1
Graph ID: 22, number of nodes: 35, label: -1
Graph ID: 23, numbe

In [4]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_bzr_distance_matrix(graph) for graph in graphs_list]

In [5]:
# Balance the dataset by selecting equal numbers from each class
class_1_indices = [i for i, label in enumerate(labels_list) if label == 1]
class_neg1_indices = [i for i, label in enumerate(labels_list) if label == -1]

# Randomly select 102 elements from class -1 to match class 1
random.seed(42)  # Set seed for reproducibility
selected_neg1_indices = random.sample(class_neg1_indices, 86)

# Combine selected indices
balanced_indices = class_1_indices + selected_neg1_indices
random.shuffle(balanced_indices)  # Shuffle to mix the classes

# Create balanced dataset
balanced_graphs_list = [graphs_list[i] for i in balanced_indices]
balanced_labels_list = [labels_list[i] for i in balanced_indices]
balanced_distance_matrices = [distance_matrices[i] for i in balanced_indices]

print(f"\nBalanced dataset:")
print("Elements of Class 1: ", balanced_labels_list.count(1))
print("Elements of Class -1: ", balanced_labels_list.count(-1))

# Update the variables to use balanced data
graphs_list = balanced_graphs_list
labels_list = balanced_labels_list
distance_matrices = balanced_distance_matrices


Balanced dataset:
Elements of Class 1:  86
Elements of Class -1:  86


In [6]:
"""fGW-NCC: alpha hyperparameter tuning"""

'fGW-NCC: alpha hyperparameter tuning'

In [7]:
### TEST 1: Use dataset average number of nodes ###
alpha_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alpha_tests = {
        alph: []
        for alph in alpha_list
    }  # dictionary to store accuracies for each k
for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    bzr_exp = fgw_barycenter_alg_alpha(distance_matrices, 
                                       graphs_list, 
                                       labels_list, 
                                       num_folds=10, 
                                       num_nodes_negone=36, 
                                       num_nodes_one=36,
                                       alpha=alpha,
                                       method="kfold")
    alpha_tests[alpha].append(bzr_exp)


Currently testing alpha = 0.0

k-fold CV selected
Fold 0:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  1.2939707328569696  Fold 0
F1 Score (macro avg): 55.00000000000001%, Fold 0
Classification Accuracy: 55.55555555555556% Fold 0
Classification Accuracy for Class -1: 66.66666666666666% Fold 0
Classification Accuracy for Class 1: 44.44444444444444% Fold 0
Fold 1:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  1.1787561085665779  Fold 1
F1 Score (macro avg): 77.5%, Fold 1
Classification Accuracy: 77.77777777777779% Fold 1
Classification Accuracy for Class -1: 85.71428571428571% Fold 1
Classification Accuracy for Class 1: 72.72727272727273% Fold 1
Fold 2:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  1.3098648009097928  Fold 2
F1 Score (macro avg): 58.245614035087726%, Fold 2
Class

In [20]:
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0
alph_avg_accuracies = {}
alph_avg_f1_scores = {}
classes = [-1, 1]

for alph in alpha_list:
        avg_acc = sum(alpha_tests[alph][0][0]) / len(alpha_tests[alph][0][0])
        avg_f1 = sum(alpha_tests[alph][0][3]) / len(alpha_tests[alph][0][3])

        alph_avg_accuracies[alph] = {
            'overall':
            avg_acc,
            'class_1':
            sum(alpha_tests[alph][0][1]) / len(alpha_tests[alph][0][1]),
            'class_-1':
            sum(alpha_tests[alph][0][2]) / len(alpha_tests[alph][0][2])
        }
        alph_avg_f1_scores[alph] = avg_f1

        print(f"\nalpha={alph} Average Overall Accuracy: {avg_acc:.2f}%")
        print(f"alpha={alph} Average F1 Score (weighted avg): {avg_f1:.2f}%")
        for class_label in [1, -1]:
            print(
                f"alpha={alph} Average Class {class_label} Accuracy: {alph_avg_accuracies[alph][f'class_{class_label}']:.2f}%"
            )

        if avg_acc > best_alpha_acc:
            best_alpha_acc = avg_acc
            best_alpha = alph

        if avg_f1 > best_alpha_f1:
            best_alpha_f1 = avg_f1

print(f"\nBest performance achieved with alpha={best_alpha}:")
print(f"Overall Accuracy: {alph_avg_accuracies[best_alpha]['overall']:.2f}%")
print(f"F1 Score (weighted avg): {alph_avg_f1_scores[best_alpha]:.2f}%")
print(f"Class 1 Accuracy: {alph_avg_accuracies[best_alpha]['class_1']:.2f}%")
print(f"Class -1 Accuracy: {alph_avg_accuracies[best_alpha]['class_-1']:.2f}%")


alpha=0.0 Average Overall Accuracy: 69.80%
alpha=0.0 Average F1 Score (weighted avg): 69.06%
alpha=0.0 Average Class 1 Accuracy: 66.38%
alpha=0.0 Average Class -1 Accuracy: 73.19%

alpha=0.1 Average Overall Accuracy: 72.12%
alpha=0.1 Average F1 Score (weighted avg): 71.43%
alpha=0.1 Average Class 1 Accuracy: 71.38%
alpha=0.1 Average Class -1 Accuracy: 73.19%

alpha=0.2 Average Overall Accuracy: 69.84%
alpha=0.2 Average F1 Score (weighted avg): 69.21%
alpha=0.2 Average Class 1 Accuracy: 71.38%
alpha=0.2 Average Class -1 Accuracy: 69.15%

alpha=0.3 Average Overall Accuracy: 70.42%
alpha=0.3 Average F1 Score (weighted avg): 69.76%
alpha=0.3 Average Class 1 Accuracy: 71.38%
alpha=0.3 Average Class -1 Accuracy: 70.26%

alpha=0.4 Average Overall Accuracy: 69.87%
alpha=0.4 Average F1 Score (weighted avg): 69.18%
alpha=0.4 Average Class 1 Accuracy: 71.21%
alpha=0.4 Average Class -1 Accuracy: 68.97%

alpha=0.5 Average Overall Accuracy: 67.52%
alpha=0.5 Average F1 Score (weighted avg): 66.56%
a

In [9]:
"""fGW-kNN: alpha hyperparameter tuning"""

'fGW-kNN: alpha hyperparameter tuning'

In [10]:
k_list = [1,2,3,4,5,6,7,8,9,10]
num_folds = 10

In [11]:
alpha_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alpha_tests_k = {
        alph: {}
        for alph in alpha_list
    }  # dictionary to store best k information for each alpha

for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    class_acc_dict, by_class_acc_dict, f1_score_dict, best_k_info = fgw_knn_alpha(distance_matrices, graphs_list, labels_list, num_folds, k_list=k_list, method="kfold",alpha=alpha)
    
    # Store the best k information for this alpha
    alpha_tests_k[alpha] = best_k_info


Currently testing alpha = 0.0

k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 55.55555555555556%, Fold 0
k=1, Class -1 Accuracy: 77.77777777777779%, Fold 0
k=1, Overall Classification Accuracy: 66.66666666666666%, Fold 0
k=1, F1 Score (macro avg): 66.25%, Fold 0
k=2, Class 1 Accuracy: 55.55555555555556%, Fold 0
k=2, Class -1 Accuracy: 77.77777777777779%, Fold 0
k=2, Overall Classification Accuracy: 66.66666666666666%, Fold 0
k=2, F1 Score (macro avg): 66.25%, Fold 0
k=3, Class 1 Accuracy: 33.33333333333333%, Fold 0
k=3, Class -1 Accuracy: 88.88888888888889%, Fold 0
k=3, Overall Classification Accuracy: 61.111111111111114%, Fold 0
k=3, F1 Score (macro avg): 57.86%, Fold 0
k=4, Class 1 Accuracy: 44.44444444444444%, Fold 0
k=4, Class -1 Accuracy: 88.88888888888889%, Fold 0
k=4, Overall Classification Accuracy: 66.66666666666666%, Fold 0
k=4, F1 Score (macro avg): 64.94%, Fold 0
k=5, Class 1 Accuracy: 44.44444444444444%, Fold 0
k=5, Class -1 Accuracy: 88.88888888888889%, Fold 0
k=5, Ov

In [12]:
# Find best alpha based on the best k performance for each alpha
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0

print("\n" + "="*60)
print("SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:")
print("="*60)

for alph in alpha_list:
    best_k_info = alpha_tests_k[alph]
    
    print(f"\nalpha={alph}:")
    print(f"  Best k: {best_k_info['best_k']}")
    print(f"  Overall Accuracy: {best_k_info['best_overall_accuracy']:.2f}%")
    print(f"  F1 Score (weighted avg): {best_k_info['best_f1_score']:.2f}%")
    print(f"  Class 1 Accuracy: {best_k_info['best_class_1_accuracy']:.2f}%")
    print(f"  Class -1 Accuracy: {best_k_info['best_class_minus1_accuracy']:.2f}%")

    # Check if this alpha gives the best overall accuracy
    if best_k_info['best_overall_accuracy'] > best_alpha_acc:
        best_alpha_acc = best_k_info['best_overall_accuracy']
        best_alpha = alph

    # Track best F1 score
    if best_k_info['best_f1_score'] > best_alpha_f1:
        best_alpha_f1 = best_k_info['best_f1_score']

print("\n" + "="*60)
print("OVERALL BEST PERFORMANCE:")
print("="*60)
best_info = alpha_tests_k[best_alpha]
print(f"Best alpha: {best_alpha}")
print(f"Best k for this alpha: {best_info['best_k']}")
print(f"Overall Accuracy: {best_info['best_overall_accuracy']:.2f}%")
print(f"F1 Score (weighted avg): {best_info['best_f1_score']:.2f}%")
print(f"Class 1 Accuracy: {best_info['best_class_1_accuracy']:.2f}%")
print(f"Class -1 Accuracy: {best_info['best_class_minus1_accuracy']:.2f}%")


SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:

alpha=0.0:
  Best k: 1
  Overall Accuracy: 79.64%
  F1 Score (weighted avg): 79.42%
  Class 1 Accuracy: 78.28%
  Class -1 Accuracy: 82.67%

alpha=0.1:
  Best k: 1
  Overall Accuracy: 79.08%
  F1 Score (weighted avg): 78.87%
  Class 1 Accuracy: 77.37%
  Class -1 Accuracy: 82.67%

alpha=0.2:
  Best k: 1
  Overall Accuracy: 79.67%
  F1 Score (weighted avg): 79.46%
  Class 1 Accuracy: 77.37%
  Class -1 Accuracy: 83.78%

alpha=0.3:
  Best k: 1
  Overall Accuracy: 79.12%
  F1 Score (weighted avg): 78.83%
  Class 1 Accuracy: 76.26%
  Class -1 Accuracy: 83.78%

alpha=0.4:
  Best k: 1
  Overall Accuracy: 79.12%
  F1 Score (weighted avg): 78.83%
  Class 1 Accuracy: 76.26%
  Class -1 Accuracy: 83.78%

alpha=0.5:
  Best k: 1
  Overall Accuracy: 79.08%
  F1 Score (weighted avg): 78.66%
  Class 1 Accuracy: 74.59%
  Class -1 Accuracy: 84.89%

alpha=0.6:
  Best k: 1
  Overall Accuracy: 79.64%
  F1 Score (weighted avg): 79.27%
  Class 1 Accuracy: 75.50%
 

In [13]:
### Standardized feature vector experiments ###
graphs_list_norm = []
labels_list_norm = []
for graph_id, graph in graphs_normalized.items():
    # Add the graph object to the list
    graphs_list_norm.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list_norm.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices_norm = [compute_bzr_distance_matrix(graph) for graph in graphs_list_norm]

In [14]:
# Balance the dataset by selecting equal numbers from each class
class_1_indices = [i for i, label in enumerate(labels_list_norm) if label == 1]
class_neg1_indices = [i for i, label in enumerate(labels_list_norm) if label == -1]

# Randomly select 102 elements from class -1 to match class 1
random.seed(42)  # Set seed for reproducibility
selected_neg1_indices = random.sample(class_neg1_indices, 86)

# Combine selected indices
balanced_indices = class_1_indices + selected_neg1_indices
random.shuffle(balanced_indices)  # Shuffle to mix the classes

# Create balanced dataset
balanced_graphs_list = [graphs_list_norm[i] for i in balanced_indices]
balanced_labels_list = [labels_list_norm[i] for i in balanced_indices]
balanced_distance_matrices = [distance_matrices_norm[i] for i in balanced_indices]

print(f"\nBalanced dataset:")
print("Elements of Class 1: ", balanced_labels_list.count(1))
print("Elements of Class -1: ", balanced_labels_list.count(-1))

# Update the variables to use balanced data
graphs_list_norm = balanced_graphs_list
labels_list_norm = balanced_labels_list
distance_matrices_norm = balanced_distance_matrices


Balanced dataset:
Elements of Class 1:  86
Elements of Class -1:  86


In [16]:
alpha_tests_norm = {
        alph: []
        for alph in alpha_list
    }  # dictionary to store accuracies for each k
for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    bzr_exp2 = fgw_barycenter_alg_alpha(distance_matrices_norm, 
                                       graphs_list_norm, 
                                       labels_list_norm, 
                                       num_folds=10, 
                                       num_nodes_negone=36, 
                                       num_nodes_one=36,
                                       alpha=alpha,
                                       method="kfold")
    alpha_tests_norm[alpha].append(bzr_exp2)


Currently testing alpha = 0.0

k-fold CV selected
Fold 0:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.3026803290908582  Fold 0
F1 Score (macro avg): 72.13622291021672%, Fold 0
Classification Accuracy: 72.22222222222221% Fold 0
Classification Accuracy for Class -1: 66.66666666666666% Fold 0
Classification Accuracy for Class 1: 77.77777777777779% Fold 0
Fold 1:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.2977070735238745  Fold 1
F1 Score (macro avg): 72.13622291021672%, Fold 1
Classification Accuracy: 72.22222222222221% Fold 1
Classification Accuracy for Class -1: 85.71428571428571% Fold 1
Classification Accuracy for Class 1: 63.63636363636363% Fold 1
Fold 2:
GW barycenter for Class -1 graphs computed
GW barycenter for Class 1 graphs computed
Distance between barycenters:  0.34037026994805464  Fold 2
F1 Score (macro avg): 52.77777777777778%,

In [17]:
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0
alph_avg_accuracies = {}
alph_avg_f1_scores = {}
classes = [-1, 1]

for alph in alpha_list:
        avg_acc = sum(alpha_tests_norm[alph][0][0]) / len(alpha_tests_norm[alph][0][0])
        avg_f1 = sum(alpha_tests_norm[alph][0][3]) / len(alpha_tests_norm[alph][0][3])

        alph_avg_accuracies[alph] = {
            'overall':
            avg_acc,
            'class_1':
            sum(alpha_tests_norm[alph][0][1]) / len(alpha_tests_norm[alph][0][1]),
            'class_-1':
            sum(alpha_tests_norm[alph][0][2]) / len(alpha_tests_norm[alph][0][2])
        }
        alph_avg_f1_scores[alph] = avg_f1

        print(f"\nalpha={alph} Average Overall Accuracy: {avg_acc:.2f}%")
        print(f"alpha={alph} Average F1 Score (weighted avg): {avg_f1:.2f}%")
        for class_label in [1, -1]:
            print(
                f"alpha={alph} Average Class {class_label} Accuracy: {alph_avg_accuracies[alph][f'class_{class_label}']:.2f}%"
            )

        if avg_acc > best_alpha_acc:
            best_alpha_acc = avg_acc
            best_alpha = alph

        if avg_f1 > best_alpha_f1:
            best_alpha_f1 = avg_f1

print(f"\nBest performance achieved with alpha={best_alpha}:")
print(f"Overall Accuracy: {alph_avg_accuracies[best_alpha]['overall']:.2f}%")
print(f"F1 Score (weighted avg): {alph_avg_f1_scores[best_alpha]:.2f}%")
print(f"Class 1 Accuracy: {alph_avg_accuracies[best_alpha]['class_1']:.2f}%")
print(f"Class -1 Accuracy: {alph_avg_accuracies[best_alpha]['class_-1']:.2f}%")


alpha=0.0 Average Overall Accuracy: 66.21%
alpha=0.0 Average F1 Score (weighted avg): 65.31%
alpha=0.0 Average Class 1 Accuracy: 66.49%
alpha=0.0 Average Class -1 Accuracy: 65.22%

alpha=0.1 Average Overall Accuracy: 66.93%
alpha=0.1 Average F1 Score (weighted avg): 66.09%
alpha=0.1 Average Class 1 Accuracy: 72.57%
alpha=0.1 Average Class -1 Accuracy: 61.49%

alpha=0.2 Average Overall Accuracy: 66.90%
alpha=0.2 Average F1 Score (weighted avg): 66.28%
alpha=0.2 Average Class 1 Accuracy: 70.55%
alpha=0.2 Average Class -1 Accuracy: 64.03%

alpha=0.3 Average Overall Accuracy: 68.10%
alpha=0.3 Average F1 Score (weighted avg): 67.11%
alpha=0.3 Average Class 1 Accuracy: 72.32%
alpha=0.3 Average Class -1 Accuracy: 64.55%

alpha=0.4 Average Overall Accuracy: 65.72%
alpha=0.4 Average F1 Score (weighted avg): 64.77%
alpha=0.4 Average Class 1 Accuracy: 69.86%
alpha=0.4 Average Class -1 Accuracy: 63.26%

alpha=0.5 Average Overall Accuracy: 65.20%
alpha=0.5 Average F1 Score (weighted avg): 63.51%
a

In [18]:
alpha_tests_k_norm = {
        alph: {}
        for alph in alpha_list
    }  # dictionary to store best k information for each alpha

for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    class_acc_dict, by_class_acc_dict, f1_score_dict, best_k_info = fgw_knn_alpha(distance_matrices_norm, 
                                                                                  graphs_list_norm, 
                                                                                  labels_list_norm, 
                                                                                  num_folds, 
                                                                                  k_list=k_list, 
                                                                                  method="kfold",alpha=alpha)
    
    # Store the best k information for this alpha
    alpha_tests_k_norm[alpha] = best_k_info


Currently testing alpha = 0.0

k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 44.44444444444444%, Fold 0
k=1, Class -1 Accuracy: 77.77777777777779%, Fold 0
k=1, Overall Classification Accuracy: 61.111111111111114%, Fold 0
k=1, F1 Score (macro avg): 60.00%, Fold 0
k=2, Class 1 Accuracy: 44.44444444444444%, Fold 0
k=2, Class -1 Accuracy: 77.77777777777779%, Fold 0
k=2, Overall Classification Accuracy: 61.111111111111114%, Fold 0
k=2, F1 Score (macro avg): 60.00%, Fold 0
k=3, Class 1 Accuracy: 33.33333333333333%, Fold 0
k=3, Class -1 Accuracy: 88.88888888888889%, Fold 0
k=3, Overall Classification Accuracy: 61.111111111111114%, Fold 0
k=3, F1 Score (macro avg): 57.86%, Fold 0
k=4, Class 1 Accuracy: 33.33333333333333%, Fold 0
k=4, Class -1 Accuracy: 88.88888888888889%, Fold 0
k=4, Overall Classification Accuracy: 61.111111111111114%, Fold 0
k=4, F1 Score (macro avg): 57.86%, Fold 0
k=5, Class 1 Accuracy: 44.44444444444444%, Fold 0
k=5, Class -1 Accuracy: 88.88888888888889%, Fold 0
k=5,

In [19]:
# Find best alpha based on the best k performance for each alpha
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0

print("\n" + "="*60)
print("SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:")
print("="*60)

for alph in alpha_list:
    best_k_info = alpha_tests_k_norm[alph]
    
    print(f"\nalpha={alph}:")
    print(f"  Best k: {best_k_info['best_k']}")
    print(f"  Overall Accuracy: {best_k_info['best_overall_accuracy']:.2f}%")
    print(f"  F1 Score (weighted avg): {best_k_info['best_f1_score']:.2f}%")
    print(f"  Class 1 Accuracy: {best_k_info['best_class_1_accuracy']:.2f}%")
    print(f"  Class -1 Accuracy: {best_k_info['best_class_minus1_accuracy']:.2f}%")

    # Check if this alpha gives the best overall accuracy
    if best_k_info['best_overall_accuracy'] > best_alpha_acc:
        best_alpha_acc = best_k_info['best_overall_accuracy']
        best_alpha = alph

    # Track best F1 score
    if best_k_info['best_f1_score'] > best_alpha_f1:
        best_alpha_f1 = best_k_info['best_f1_score']

print("\n" + "="*60)
print("OVERALL BEST PERFORMANCE:")
print("="*60)
best_info = alpha_tests_k_norm[best_alpha]
print(f"Best alpha: {best_alpha}")
print(f"Best k for this alpha: {best_info['best_k']}")
print(f"Overall Accuracy: {best_info['best_overall_accuracy']:.2f}%")
print(f"F1 Score (weighted avg): {best_info['best_f1_score']:.2f}%")
print(f"Class 1 Accuracy: {best_info['best_class_1_accuracy']:.2f}%")
print(f"Class -1 Accuracy: {best_info['best_class_minus1_accuracy']:.2f}%")


SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:

alpha=0.0:
  Best k: 1
  Overall Accuracy: 79.12%
  F1 Score (weighted avg): 78.69%
  Class 1 Accuracy: 75.42%
  Class -1 Accuracy: 83.26%

alpha=0.1:
  Best k: 1
  Overall Accuracy: 79.15%
  F1 Score (weighted avg): 78.84%
  Class 1 Accuracy: 77.37%
  Class -1 Accuracy: 82.15%

alpha=0.2:
  Best k: 1
  Overall Accuracy: 77.42%
  F1 Score (weighted avg): 77.03%
  Class 1 Accuracy: 73.89%
  Class -1 Accuracy: 82.15%

alpha=0.3:
  Best k: 1
  Overall Accuracy: 77.97%
  F1 Score (weighted avg): 77.57%
  Class 1 Accuracy: 75.91%
  Class -1 Accuracy: 80.90%

alpha=0.4:
  Best k: 1
  Overall Accuracy: 77.35%
  F1 Score (weighted avg): 77.05%
  Class 1 Accuracy: 74.80%
  Class -1 Accuracy: 81.42%

alpha=0.5:
  Best k: 1
  Overall Accuracy: 80.16%
  F1 Score (weighted avg): 79.83%
  Class 1 Accuracy: 77.41%
  Class -1 Accuracy: 83.64%

alpha=0.6:
  Best k: 1
  Overall Accuracy: 78.99%
  F1 Score (weighted avg): 78.54%
  Class 1 Accuracy: 73.64%
 